# 🔬 Does Your Cancer Classifier Actually Work? BreakHis, Done Honestly

**BreakHis 400x** (2025 upload) - H&E breast-tissue microscopy at 400x, **benign vs malignant**.

Most BreakHis notebooks quietly cheat: every patient contributes *dozens* of patches, so a random
split drops the **same patient** into train **and** test and the accuracy balloons. We do two things
properly instead:

1. **🔒 Patient-disjoint split** - parse the patient/slide ID from each filename (`GroupShuffleSplit`),
   so no patient appears on both sides.
2. **🗳️ Patient-level voting** - a diagnosis is made *per patient* by pooling that patient's patches,
   which is how a pathologist actually reports.

We report the honest patch-level *and* patient-level scores, and use **Grad-CAM** to check the model
reads tissue morphology, not stain artefacts. **GPU-only.** If the honesty is useful, an upvote helps 🙏

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q grad-cam timm

## Setup - force GPU + auto-locate data

In [ ]:
import os, re
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, balanced_accuracy_score, roc_auc_score)

assert torch.cuda.is_available(), 'GPU required'
_ = nn.Conv2d(3,4,3).cuda()(torch.randn(1,3,8,8,device='cuda')); torch.cuda.synchronize()
DEVICE = torch.device('cuda'); print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

def find_root():
    for base in ['/kaggle/input', os.path.expanduser('~/.cache/kagglehub'), os.path.expanduser('~/data_kaggle')]:
        b = Path(base)
        if not b.exists(): continue
        for d in b.rglob('train'):
            if (d/'benign').exists() and (d.parent/'test').exists():
                return d.parent
    raise FileNotFoundError('BreakHis train/test folders not found')
ROOT = find_root(); print('data root:', ROOT)
CLASSES = ['benign', 'malignant']

## 1. Scan images, parse patient IDs, and measure the leak

BreakHis filenames look like `SOB_B_A-14-**22549CD**-400-006.png` - the bold field is the
**patient/slide ID**. We pool the vendor's `train` and `test` folders, then check how many
patients appear in *both*. Any overlap = leakage.

In [ ]:
def patient_id(path):
    # SOB_<B|M>_<subtype>-<year>-<PATIENT>-<mag>-<seq>.png  ->  <PATIENT>
    m = re.match(r'SOB_[BM]_[A-Z]+-\d+-([A-Za-z0-9]+)-', Path(path).name)
    return m.group(1) if m else Path(path).stem

def scan(split):
    items = []
    for lbl, c in enumerate(CLASSES):
        for p in (ROOT/split/c).glob('*.png'):
            items.append((str(p), lbl, patient_id(p)))
    return items

vendor_train, vendor_test = scan('train'), scan('test')
pt_tr = {pid for _,_,pid in vendor_train}; pt_te = {pid for _,_,pid in vendor_test}
overlap = pt_tr & pt_te
print(f'vendor split: train={len(vendor_train)} imgs / {len(pt_tr)} patients, '
      f'test={len(vendor_test)} imgs / {len(pt_te)} patients')
print(f'patients in BOTH vendor train & test (leak): {len(overlap)}')

allitems = vendor_train + vendor_test
from collections import Counter
cnt = Counter(CLASSES[y] for _,y,_ in allitems)
print('total images:', len(allitems), '| class counts:', dict(cnt),
      '| unique patients:', len({p for _,_,p in allitems}))
plt.figure(figsize=(6,3)); plt.bar(list(cnt.keys()), list(cnt.values()), color=['#55a868','#c44e52'])
plt.title('BreakHis 400x - class balance (benign vs malignant)'); plt.show()

## 2. Build a **patient-disjoint** split

We ignore the vendor split and re-partition with `GroupShuffleSplit` grouped by patient ID, so
no patient's tissue is in both train and test. This is the split that gives an honest score.

In [ ]:
paths = np.array([p for p,_,_ in allitems])
labels = np.array([y for _,y,_ in allitems])
groups = np.array([g for _,_,g in allitems])
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(paths, labels, groups))
assert not (set(groups[tr_idx]) & set(groups[te_idx])), 'split leaked a patient!'
tr = list(zip(paths[tr_idx], labels[tr_idx]))
te = list(zip(paths[te_idx], labels[te_idx], groups[te_idx]))   # keep patient id for voting
print(f'patient-disjoint: train={len(tr)} imgs / {len(set(groups[tr_idx]))} patients | '
      f'test={len(te)} imgs / {len(set(groups[te_idx]))} patients')

MEAN=torch.tensor([0.485,0.456,0.406]).view(3,1,1); STD=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
class Histo(Dataset):
    def __init__(self, items, train=False): self.items, self.train = list(items), train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        rec = self.items[i]; p,y = rec[0], rec[1]
        img = cv2.resize(cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB),(224,224))
        if self.train:
            if np.random.rand()<0.5: img=img[:,::-1]
            if np.random.rand()<0.5: img=img[::-1,:]
        x = torch.from_numpy(np.ascontiguousarray(img)).permute(2,0,1).float()/255.
        return (x-MEAN)/STD, int(y)
dl_tr=DataLoader(Histo(tr,True),32,shuffle=True,num_workers=2,pin_memory=True)
dl_te=DataLoader(Histo(te),32,num_workers=2,pin_memory=True)

# a peek at real patches with their labels
fig,axes=plt.subplots(2,4,figsize=(13,7)); ds=Histo(te)
for ax,idx in zip(axes.ravel(), np.linspace(0,len(ds)-1,8).astype(int)):
    x,y=ds[idx]; rgb=np.clip(x.permute(1,2,0).numpy()*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406]),0,1)
    ax.imshow(rgb); ax.set_title(CLASSES[y],fontsize=9); ax.axis('off')
plt.suptitle('BreakHis 400x H&E patches (held-out patients)'); plt.tight_layout(); plt.show()

## 3. Train EfficientNet-b0 (class-weighted for the benign/malignant imbalance)

In [ ]:
w = torch.tensor([len(labels)/(2*(labels==0).sum()), len(labels)/(2*(labels==1).sum())],
                 dtype=torch.float32, device=DEVICE)
model=timm.create_model('efficientnet_b0',pretrained=True,num_classes=2).to(DEVICE)
crit=nn.CrossEntropyLoss(weight=w); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-5)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=15); scaler=torch.cuda.amp.GradScaler()
for ep in range(15):
    model.train(); tot=0
    for x,y in dl_tr:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
        with torch.autocast('cuda'): loss=crit(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot+=loss.item()
    sched.step(); print(f'epoch {ep+1}/15 loss={tot/len(dl_tr):.4f}')

## 4. Evaluate - patch-level, then **patient-level voting**

In [ ]:
# per-patch probabilities on the held-out patients
model.eval(); probs=[]
dl_eval=DataLoader(Histo(te),64,num_workers=2,pin_memory=True)
with torch.inference_mode():
    for x,_ in dl_eval: probs += torch.softmax(model(x.to(DEVICE)),1)[:,1].cpu().tolist()
probs=np.array(probs); ys=np.array([r[1] for r in te]); ppat=np.array([r[2] for r in te])
patch_pred=(probs>0.5).astype(int)
p_acc=(patch_pred==ys).mean(); p_bacc=balanced_accuracy_score(ys,patch_pred); p_auc=roc_auc_score(ys,probs)
print(f'PATCH-LEVEL   acc={p_acc:.3f}  balanced_acc={p_bacc:.3f}  AUC={p_auc:.3f}')

# patient-level: average the patches of each patient, then one diagnosis per patient
pat_y, pat_prob = [], []
for pid in np.unique(ppat):
    mask = ppat==pid; pat_y.append(int(round(ys[mask].mean()))); pat_prob.append(probs[mask].mean())
pat_y=np.array(pat_y); pat_prob=np.array(pat_prob); pat_pred=(pat_prob>0.5).astype(int)
v_acc=(pat_pred==pat_y).mean(); v_bacc=balanced_accuracy_score(pat_y,pat_pred); v_auc=roc_auc_score(pat_y,pat_prob)
print(f'PATIENT-LEVEL acc={v_acc:.3f}  balanced_acc={v_bacc:.3f}  AUC={v_auc:.3f}  ({len(pat_y)} patients)')
print(); print(classification_report(pat_y,pat_pred,target_names=CLASSES))

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].bar(['patch\nacc','patch\nAUC','patient\nacc','patient\nAUC'],[p_acc,p_auc,v_acc,v_auc],
          color=['#bbb','#999','#c44e52','#8c2d34'])
for i,v in enumerate([p_acc,p_auc,v_acc,v_auc]): ax[0].text(i,v+0.01,f'{v:.2f}',ha='center',fontweight='bold')
ax[0].set_ylim(0,1.05); ax[0].set_title('Voting lifts the honest score')
ConfusionMatrixDisplay(confusion_matrix(pat_y,pat_pred),display_labels=CLASSES).plot(cmap='Blues',ax=ax[1],colorbar=False)
ax[1].set_title(f'Patient-level (acc={v_acc:.3f}, AUC={v_auc:.3f})')
plt.tight_layout(); plt.show()

## 5. 🔍 Grad-CAM - is the model reading the tissue?

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
layer=[m for m in model.modules() if isinstance(m,nn.Conv2d)][-1]; ds=Histo(te)
fig,axes=plt.subplots(2,4,figsize=(13,7))
with GradCAM(model=model,target_layers=[layer]) as cam:
    for ax,idx in zip(axes.ravel(), np.linspace(0,len(ds)-1,8).astype(int)):
        x,y=ds[idx]
        rgb=np.clip(x.permute(1,2,0).numpy()*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406]),0,1).astype(np.float32)
        ax.imshow(show_cam_on_image(rgb, cam(input_tensor=x[None].to(DEVICE))[0], use_rgb=True))
        ax.set_title(CLASSES[y],fontsize=9); ax.axis('off')
plt.suptitle('Grad-CAM on BreakHis 400x - attention over tissue morphology'); plt.tight_layout(); plt.show()

## 🏁 Takeaways

Two rules turn a vanity number into an honest one: **split by patient** (no leakage) and **diagnose per
patient** (voting over a patient's patches). The patient-level accuracy/AUC above is what actually predicts
performance on a *new* patient - the number that matters clinically - and Grad-CAM confirms the model attends
to nuclear/tissue morphology. Next: multi-magnification fusion (40x-400x), stain normalisation, and an
attention-MIL head that pools patches inside the network. *Thanks - upvotes welcome! 🙌*